# Transformer XRD
This notebook is adapted from [VIT_Materials_Spectra](https://github.com/linresearchgroup/ViT_Materials_Spectra) and uses vision transformers to classify space groups from XRD data.

- [Data](https://drive.google.com/drive/folders/ANONYMIZED-DATA-FOLDER)
- [Models](https://drive.google.com/drive/folders/ANONYMIZED-MODELS-FOLDER)

Adaptations from standard ViTs:

| Feature           | Standard ViT (Images) | This ViT Model (Spectra) |
|------------------|----------------------|--------------------------|
| Input Data Type  | 2D Images (H × W × C) | 1D Spectral Data (XRD)   |
| Tokenization     | Image patches         | Spectral segments        |
| Patch Embedding  | Linear Projection     | `Spectra_Embedding` function |
| Positional Encoding | Learned            | Learned (`pos_embed`)    |
| Output          | Image classification   | XRD classification/regression |


## Imports

In [ ]:
from functools import partial
from netrc import netrc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Sequential, Linear, ReLU, GRU

from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
#from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd
import pickle

# Import the library to mount Google Drive
#from google.colab import drive

import h5py

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

from torchviz import make_dot
from torchinfo import summary
import intel_extension_for_pytorch as ipex

import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger, WandbLogger, MLFlowLogger
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.profilers import PyTorchProfiler

import wandb

from pytorch_lightning.callbacks import EarlyStopping, Callback
from pytorch_lightning.callbacks import ModelCheckpoint
from torch.profiler import profile, record_function, ProfilerActivity
from pytorch_lightning.callbacks import TQDMProgressBar
from pytorch_lightning.callbacks import RichProgressBar
from pytorch_lightning.strategies import DDPStrategy

import os


In [ ]:
print(torch.__version__)
print(ipex.__version__)

## Global Variables

In [ ]:
datasource = "PowCOD"
size = "small"
intensity_points = 500
twotheta = True
space = "2theta"
twotheta_start = 4.0
twotheta_end = 44.0
test_split = 0.20
validation_split = 0.20
num_classes = 230
model_path = '/scratch/10486/UserB/Models/'
data_path = '/scratch/10486/UserB/Data/'
root_path = '/scratch/10486/UserB/'

In [ ]:
num_epochs = 10
batch_size = 64
embed_dim=100
depth=11
num_heads=2
mlp_ratio=4.0

In [ ]:
wandb_project = "XRD_SpaceGroup_Classification_ViT"

In [ ]:
if hasattr(torch, "xpu") and torch.xpu.is_available():
    print("Intel GPU (XPU) is available.")
    print("PyTorch version:", torch.__version__)
    print("Device:", torch.xpu.get_device_name(0))
else:
    print("No Intel GPU detected.")

In [ ]:
torch.xpu.is_available()


In [ ]:
#with open('....pkl', 'rb') as f:
#    Data = pickle.load(f)

In [ ]:
#Reference:https://github.com/chensaian/TransG-Net

## Functions

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self,
                 dim,
                 num_heads,
                 mlp_ratio=4.,
                 qkv_bias=False,
                 qk_scale=None,
                 drop_ratio=0.,
                 attn_drop_ratio=0.,
                 drop_path_ratio=0.,
                 act_layer=nn.GELU,
                 norm_layer=nn.LayerNorm):
        super(EncoderBlock, self).__init__()
        self.norm1 = norm_layer(dim)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias, qk_scale=qk_scale,
                              attn_drop_ratio=attn_drop_ratio, proj_drop_ratio=drop_ratio)
        self.drop_path = DropPath(
            drop_path_ratio) if drop_path_ratio > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim,
                       act_layer=act_layer, drop=drop_ratio)

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

In [ ]:
class Attention(nn.Module):

    def __init__(self,
                 dim,
                 num_heads=2,
                 qkv_bias=False,
                 qk_scale=None,
                 attn_drop_ratio=0.,
                 proj_drop_ratio=0.):
        super(Attention, self).__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop_ratio)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop_ratio)

    def forward(self, x):
        # [batch_size, num_patches + 1, total_embed_dim]
        B, N, C = x.shape
        # qkv(): -> [batch_size, num_patches + 1, 3 * total_embed_dim]
        # reshape: -> [batch_size, num_patches + 1, 3, num_heads, embed_dim_per_head]
        # permute: -> [3, batch_size, num_heads, num_patches + 1, embed_dim_per_head]
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        # [batch_size, num_heads, num_patches + 1, embed_dim_per_head]
        # make torchscript happy (cannot use tensor as tuple)
        q, k, v = qkv[0], qkv[1], qkv[2]
        # transpose: -> [batch_size, num_heads, embed_dim_per_head, num_patches + 1]
        # @: multiply -> [batch_size, num_heads, num_patches + 1, num_patches + 1]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        # @: multiply -> [batch_size, num_heads, num_patches + 1, embed_dim_per_head]
        # transpose: -> [batch_size, num_patches + 1, num_heads, embed_dim_per_head]
        # reshape: -> [batch_size, num_patches + 1, total_embed_dim]
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


In [ ]:
class Mlp(nn.Module):

    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

In [ ]:
import torch.nn.functional as F

def Spectra_Embedding(x, spec_length, embed_dim):
    batch_size = x.shape[0]
    new_spec_length = (spec_length // embed_dim) * embed_dim
    x = x[:, :new_spec_length]
    x = torch.reshape(x, (batch_size, spec_length // embed_dim, embed_dim))
    return x
'''
def Spectra_Embedding_old(x, spec_length, embed_dim):

    batch_size = x.shape[0]
    x = torch.reshape(x, (batch_size, spec_length // embed_dim, embed_dim))
    return x

def Spectra_Embedding_enlong(x, spec_length, embed_dim):
    batch_size = x.shape[0]
    remainder = spec_length % embed_dim
    if remainder != 0:
        pad = embed_dim - remainder
        x = F.pad(x, (0, pad))
    x = torch.reshape(x, (batch_size, -1, embed_dim))
    return x
'''

In [ ]:
class VIT(nn.Module):
    def __init__(self, spec_length=2000, num_output=1,
                 embed_dim=40, depth=12, num_heads=2, mlp_ratio=4.0, qkv_bias=True,
                 qk_scale=None, drop_ratio=0.,
                 attn_drop_ratio=0., drop_path_ratio=0., norm_layer=None,
                 act_layer=None):

        # MSTransformer
        super(VIT, self).__init__()
        self.num_classes = num_output
        self.spec_length = spec_length
        self.num_features = self.embed_dim = embed_dim
        norm_layer = norm_layer or partial(nn.LayerNorm, eps=1e-6)
        act_layer = act_layer or nn.GELU
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, (spec_length//embed_dim) + 1, embed_dim))
        self.pos_drop = nn.Dropout(p=drop_ratio)
        dpr = [x.item() for x in torch.linspace(0, drop_path_ratio, depth)]
        self.blocks = nn.Sequential(*[
            EncoderBlock(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                        drop_ratio=drop_ratio, attn_drop_ratio=attn_drop_ratio, drop_path_ratio=dpr[i],
                        norm_layer=norm_layer, act_layer=act_layer)
            for i in range(depth)
        ])
        self.norm = norm_layer(embed_dim)

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(_init_vit_weights)

        self.head = nn.Linear(embed_dim, num_output)

    def forward(self, x):
        # [B , xrd_length] --> [B , xrd_length/embed_dim , embed_dim]
        x = Spectra_Embedding(x, self.spec_length, self.embed_dim)
        cls_token = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = self.pos_drop(x + self.pos_embed)
        x = self.blocks(x)
        x = self.norm(x)
        x = self.head(x)

        return x[:, 0]

def _init_vit_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.trunc_normal_(m.weight, std=.01)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.LayerNorm):
        nn.init.zeros_(m.bias)
        nn.init.ones_(m.weight)

def VIT_model(spec_length=2251,num_output: int = 1, embed_dim=40, depth=12, num_heads=2, mlp_ratio=4.0, drop_ratio = 0.):
    model = VIT(spec_length=spec_length,
                              embed_dim=embed_dim,
                              depth=depth,
                              num_heads=num_heads,
                              num_output=num_output,
                              mlp_ratio=mlp_ratio,
                              drop_ratio=drop_ratio)
    return model


## Data Loader

In [ ]:
!ls ../Data

In [ ]:
with h5py.File(f"{data_path}{datasource}_{size}_{intensity_points}_{space}_{twotheta_start}_{twotheta_end}_{test_split}_{validation_split}.h5", "r") as f:
    X_train = f["X_train"][:]
    y_train = f["y_train"][:]
    X_val = f["X_val"][:]
    y_val = f["y_val"][:]
    X_test = f["X_test"][:]
    y_test = f["y_test"][:]

print("Data successfully loaded!")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")


## Training Dataset

In [ ]:
# Cross entropy loss requires 0 indices

y_train = y_train - 1  # Convert range 1-230 → 0-229
y_test = y_test - 1
y_val = y_val - 1

In [ ]:

X_train = torch.tensor(X_train, dtype=torch.float)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.float)
y_test = torch.tensor(y_test, dtype=torch.long)
X_val = torch.tensor(X_val, dtype=torch.float)
y_val = torch.tensor(y_val, dtype=torch.long)



In [ ]:
# no encoding necessary
#label_encoder = LabelEncoder()
#y = label_encoder.fit_transform(y)


## Lightning Modeling

### WandB Configuration

In [ ]:
# Initialize W&B
wandb.login()

# Define a config dictionary object
config = {
    "num_epochs": num_epochs,
    "batch_size": batch_size,
    "embed_dim": embed_dim,
    "depth": depth,
    "num_heads": num_heads,
    "mlp_ratio": mlp_ratio,
}

# Define Logger
wandb_logger = WandbLogger(project=wandb_project, config=config)

In [ ]:
# Check XPU availability
device = "xpu" if torch.xpu.is_available() else "cpu"


## Attention Mapping

In [ ]:
# ✅ Custom Saliency Mapping for XRD Spectra
def compute_saliency(model, x_sample):
    model.eval()
    x_sample.requires_grad_()
    
    output = model(x_sample)  # Forward pass
    target_class = output.argmax(dim=1)  # Get predicted class

    output[0, target_class].backward()  # Compute gradients
    saliency = x_sample.grad.abs().cpu().numpy()  # Absolute gradient values

    return saliency

class SaliencyLogger(Callback):
    def on_validation_epoch_end(self, trainer, pl_module):
        sample_x, _ = next(iter(val_loader))  # Get a sample batch
        sample_x = sample_x.to(device)[:1]  # Select one spectrum

        saliency = compute_saliency(pl_module, sample_x)

        # Plot Saliency Map
        plt.figure(figsize=(10, 4))
        plt.plot(saliency[0], label="Saliency")
        plt.xlabel("Spectral Position")
        plt.ylabel("Importance")
        plt.legend()
        plt.title("Saliency Map for XRD Spectrum")

        # Save & Log to WandB
        wandb.log({"Saliency Map": wandb.Image(plt)})
        plt.close()

In [ ]:
# Model checkpointing callback
checkpoint_callback = ModelCheckpoint(
    dirpath="{root}Models/checkpoints/",
    filename="xrd_model-{epoch:02d}-{val_loss:.2f}",
    save_top_k=3,  # Save top 3 best models
    monitor="val_loss",
    mode="min",
    save_weights_only=True
)

## Lightning Model

In [ ]:
import torch
import torch.nn as nn
import pytorch_lightning as pl

class VITLightningModel(pl.LightningModule):
    def __init__(self, model):
        super(VITLightningModel, self).__init__()
        self.model = model  # Pass the VIT model directly to LightningModel
        self.loss_func = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.model(x)

    def compute_topk_accuracy(self, outputs, targets, ks=[1, 3, 5]):
        """Computes top-k accuracy for given k values."""
        _, pred = outputs.topk(max(ks), dim=1, largest=True, sorted=True)  # Get top-k predictions
        correct = pred.eq(targets.view(-1, 1).expand_as(pred))  # Compare predictions with targets
        
        return {f"top{k}_acc": correct[:, :k].sum().item() for k in ks}  # Compute accuracy

    def training_step(self, batch, batch_idx):
        x, y = batch
        x, y = x.to(self.device), y.to(self.device)
        outputs = self.model(x)
        loss = self.loss_func(outputs, y)

        # Compute top-k accuracies
        topk_accs = self.compute_topk_accuracy(outputs, y)

        # Log metrics
        self.log_dict({
            "train_loss": loss,
            "train_top1_acc": topk_accs["top1_acc"] / y.size(0),
            "train_top3_acc": topk_accs["top3_acc"] / y.size(0),
            "train_top5_acc": topk_accs["top5_acc"] / y.size(0),
        }, prog_bar=True, logger=True, on_step=True, on_epoch=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        x, y = x.to(self.device), y.to(self.device)
        outputs = self.model(x)
        loss = self.loss_func(outputs, y)

        # Compute top-k accuracies
        topk_accs = self.compute_topk_accuracy(outputs, y)

        # Log metrics
        self.log_dict({
            "val_loss": loss,
            "val_top1_acc": topk_accs["top1_acc"] / y.size(0),
            "val_top3_acc": topk_accs["top3_acc"] / y.size(0),
            "val_top5_acc": topk_accs["top5_acc"] / y.size(0),
        }, prog_bar=True, logger=True, on_epoch=True)

        return loss
    
    
    def test_step(self, batch, batch_idx):
        x, y = batch
        x, y = x.to(self.device), y.to(self.device)
        outputs = self.model(x)
        loss = self.loss_func(outputs, y)

        # Compute top-k accuracies
        topk_accs = self.compute_topk_accuracy(outputs, y)

        # Log test metrics
        self.log_dict({
            "test_loss": loss,
            "test_top1_acc": topk_accs["top1_acc"] / y.size(0),
            "test_top3_acc": topk_accs["top3_acc"] / y.size(0),
            "test_top5_acc": topk_accs["top5_acc"] / y.size(0),
        }, prog_bar=True, logger=True)

        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.model.parameters(), lr=1e-4)

### Training

In [ ]:
torch.cuda.empty_cache()

In [ ]:
# Set WandB API URL and port explicitly if you receive any port contention issues
!export WANDB_API_URL="http://localhost:21949"  # Change 19949 to an available port

In [ ]:

# Check if the system supports XPU (Intel GPUs)
if torch.xpu.is_available():
    # Get the number of XPU devices available
    num_xpus = torch.xpu.device_count()
    print(f"Number of XPU devices available: {num_xpus}")
else:
    print("XPU is not available. No Intel GPUs detected.")

In [ ]:

# Sweep configuration
sweep_config = {
    "method": "bayes",  # Bayesian Optimization for best performance
    "metric": {"name": "val_loss", "goal": "minimize"},
    "parameters": {
        "num_epochs": {"value": 10},  # Keeping fixed
        "num_devices": {"value": 1},  # Keeping fixed
        "num_precision": {"value": 16},  # Keeping fixed
        "batch_size": {"values": [32,64]},  
        "embed_dim": {"values": [64, 96, 128]},  # Ensure compatibility with num_heads
        "depth": {"values": [6, 8, 11, 14]},  
        "num_heads": {
            "values": [h for h in [2, 4, 8] if all(e % h == 0 for e in [64, 96, 128])]
        },  
        "mlp_ratio": {"values": [2.0, 4.0, 6.0]},  
        "learning_rate": {"min": 1e-5, "max": 1e-3},  
        "dropout": {"values": [0.1, 0.3, 0.5]},  
        "weight_decay": {"values": [1e-4, 1e-3, 1e-2]},  
        "optimizer": {"values": ["Adam", "SGD"]},  
#        "activation_fn": {"values": ["ReLU", "GELU"]}  
    },
}

# Initialize WandB sweep
#sweep_id = wandb.sweep(sweep_config, project="XRD_SpaceGroup_Classification_ViT")


In [ ]:
# adjust dataloader based on wandb config

#batch_size = 64
#train_data = TensorDataset(X_train, y_train)
#train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
#val_data = TensorDataset(X_val, y_val)
#val_loader = DataLoader(train_data, batch_size=batch_size, shuffle=False)
#test_data = TensorDataset(X_test, y_test)
#test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)


# Create TensorDatasets
train_data = TensorDataset(X_train, y_train)
val_data = TensorDataset(X_val, y_val)
test_data = TensorDataset(X_test, y_test)

# Intel Optimization: Use `ipex.optimize` and adjust num_workers
num_workers = 23  # Adjust based on system capabilities
pin_memory = False  # Intel GPUs do not require pinned memory

# Create DataLoaders optimized for Intel hardware
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory, persistent_workers=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory, persistent_workers=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory, persistent_workers=True)


In [ ]:
# in case of any running jobs
wandb.finish()

In [ ]:
#!torch.cuda.reset_peak_memory_stats()


In [ ]:
# Initialize WandB sweep
sweep_id = wandb.sweep(sweep_config, project=wandb_project)
def train():
    # Initialize WandB run
    with wandb.init() as run:
        config = wandb.config  # Get hyperparameters from sweep

        # Define VIT model directly
        model = VIT_model(
            spec_length=intensity_points, 
            num_output=num_classes, 
            embed_dim=config.embed_dim, 
            depth=config.depth,
            num_heads=config.num_heads,
            mlp_ratio=config.mlp_ratio,
            drop_ratio = config.dropout,
        ).to(device)

        # Define Lightning Model
        lightning_model = VITLightningModel(model)

        # Optimizer selection
        optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay) \
            if config.optimizer == "Adam" else torch.optim.SGD(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)

        # Apply IPEX optimization
        model, optimizer = ipex.optimize(model, optimizer=optimizer, dtype=torch.float32)

        # Callbacks
        checkpoint_callback = ModelCheckpoint(
            dirpath="{root_path}Models/checkpoints/",
            filename="xrd_model-{epoch:02d}-{val_loss:.2f}",
            save_top_k=3,
            monitor="val_top5_acc",
            mode="max",
            save_weights_only=True
        )
        early_stop_callback = EarlyStopping(monitor="val_loss", patience=3, mode="min")

        # WandB Logger
        wandb_logger = WandbLogger(project=wandb_project, config=config)

        # PyTorch Lightning Trainer
        trainer = pl.Trainer(
            accelerator="auto",
            devices=config.num_devices,
            precision=config.num_precision,  # Mixed precision (optional)
            max_epochs=config.num_epochs,
            logger=wandb_logger,
            callbacks=[checkpoint_callback, early_stop_callback, RichProgressBar()],
            enable_progress_bar=True,
#            strategy="ddp_notebook",  # Use ddp_notebook for notebook compatibility, currently causes memory issues
        )

        trainer.fit(lightning_model, train_dataloaders=train_loader, val_dataloaders=val_loader)

        # Save Best Model
        model_path = f"{root_path}Models/xrd_model_{run.name}.pth"
        torch.save(model.state_dict(), model_path)
        wandb.save(model_path)

        # Evaluate on Test Set
        test_results = trainer.test(lightning_model, dataloaders=test_loader)
        
        # Log Test Results to WandB
        wandb.log({
            "test_top1_acc": test_results[0].get("test_top1_acc", 0),  
            "test_top3_acc": test_results[0].get("test_top3_acc", 0),  
            "test_top5_acc": test_results[0].get("test_top5_acc", 0),  
            "test_loss": test_results[0].get("test_loss", 0),
        })

# Run sweep with multiple experiments
wandb.agent(sweep_id, function=train, count=10)  # Run 10 trials


## PyTorch Lightning Model Saving & Loading

### Save XPU Optimized Lightning Model 

In [ ]:
import intel_extension_for_pytorch as ipex

# Ensure model is in evaluation mode
model.eval()

# Optimize model for XPU before saving
optimized_model = ipex.optimize(model, dtype=torch.float32)  # or torch.bfloat16 if using lower precision

# Save the optimized model
torch.jit.save(torch.jit.script(optimized_model), f"{model_path}{datasource}_{intensity_points}_{twotheta}_{space}_{num_epochs}_{batch_size}_{embed_dim}_{num_heads}{mlp_ratio}_model_xpu_scripted.pt")

# Save standard PyTorch checkpoint
torch.save(model.state_dict(), f"{model_path}{datasource}_{intensity_points}_{twotheta}_{space}_{num_epochs}_{batch_size}_{embed_dim}_{num_heads}{mlp_ratio}_xrd_model_xpu.pth")

print("XPU-optimized model saved successfully!")

### Load from XPU Optimized Lightning Model

In [ ]:
# Compute confusion matrix, normalized to account for class imbalance
conf_matrix = confusion_matrix(all_labels, all_predicted_top1)

# Normalize by row (each true label's sum)
conf_matrix_normalized = conf_matrix.astype('float') / conf_matrix.sum(axis=1, keepdims=True)

# Plot heatmap with better contrast
plt.figure(figsize=(20, 12))
sns.heatmap(conf_matrix_normalized, cmap="Blues", annot=False, fmt=".2f", vmin=0, vmax=1)

plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Normalized Confusion Matrix Heatmap for Space Groups")
plt.show()

In [ ]:
# Load and re-optimize for XPU
model = XRDClassifier(spec_length=intensity_points, num_output=num_classes, embed_dim=embed_dim, depth=depth, num_heads=num_heads)
model.load_state_dict(torch.load("xrd_model_xpu.pth"))
model.to("xpu")  # Ensure it's on XPU

# Optimize for XPU
import intel_extension_for_pytorch as ipex
model = ipex.optimize(model, dtype=torch.float32)

print("XPU-optimized model loaded successfully!")

## PyTorch Model

In [ ]:
import datetime
import os
import time

# nvidia
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# intel
device = "xpu" if torch.xpu.is_available() else "cpu"
print(device)

# num_classes=2000
# model = VIT_model(spec_length=2251,num_output=num_classes)

# num_classes=225
model = VIT_model(spec_length=intensity_points,num_output=num_classes, embed_dim=embed_dim, depth=depth, num_heads=num_heads)
# model = VIT_model(spec_length=2251,num_output=num_classes
model = model.to(device)

optimizer = Adam(model.parameters())
loss_func = CrossEntropyLoss()

import matplotlib.pyplot as plt

num_epochs = num_epochs
losses = []
early_stop_loss = 0.98
no_improve_epoch = 0
patience = 3
epoch_times = []
total_time=0
for epoch in range(num_epochs):
    start_time = time.time()
    model.train()
    running_loss = 0.0
    for i, (x_batch, y_batch) in enumerate(train_loader):
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = loss_func(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    end_time = time.time()
    epoch_duration = end_time - start_time
    epoch_times.append(epoch_duration)
    total_time+=epoch_duration

    epoch_loss = running_loss / len(train_loader)
    print(f'Epoch: {epoch+1}, Loss: {epoch_loss}, Time: {epoch_duration:.2f}s,Total time:{total_time}')

    losses.append(epoch_loss)
    #print(f'Epoch: {epoch+1}, Loss: {epoch_loss}')


    if epoch > 0 and epoch_loss > losses[epoch-1]*early_stop_loss:
        no_improve_epoch += 1
        print(no_improve_epoch)
    else:
        no_improve_epoch = 0

    if no_improve_epoch > patience:
        print("Early stopping!")
        break

current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")



In [ ]:
# Dataloaders
#train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
#val_loader = DataLoader(val_dataset, batch_size=32)
#test_loader = DataLoader(test_dataset, batch_size=32)

# Loggers
#tensorboard_logger = TensorBoardLogger("logs/", name="XRD_experiment")
#wandb_logger = WandbLogger(project="XRD_SpaceGroup_Classification")
#mlflow_logger = MLFlowLogger(experiment_name="XRD_Attention_Logs")

# Basic Profiler Setup (without Kineto)
profiler = profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.XPU],  # XPU activity for profiling
    schedule=torch.profiler.schedule(wait=1, warmup=1, active=3, repeat=1),
    on_trace_ready=torch.profiler.tensorboard_trace_handler("profiler_logs"),  # TensorBoard logging for trace
    record_shapes=True,  # Log tensor shapes to profile
    with_stack=True  # Log the call stack for performance issues
)

# Callbacks
early_stop_callback = EarlyStopping(monitor="val_loss", patience=3, mode="min")

# Initialize Model
model = XRDClassifier(
    spec_length=intensity_points,
    num_output=num_classes,
    embed_dim=embed_dim,
    depth=depth,
    num_heads=num_heads,
).to(device)

# Optimize Model with IPEX
model, optimizer = ipex.optimize(model, optimizer=Adam(model.parameters()))

# Trainer with XPU acceleration
# Training Loop with Profiler
with profiler:
    trainer = pl.Trainer(
        accelerator="auto",
        devices=1,
        max_epochs=20,
        logger=[wandb_logger],
        callbacks=[early_stop_callback, checkpoint_callback],
        profiler="simple"
    )
    
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)



# Train
#trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

# Test
trainer.test(model, dataloaders=test_loader)

In [ ]:
# num_classes=225
# Generate dummy input
x = torch.randn(1, intensity_points)  # Batch size = 1, Sequence length = 2251
# Get only the prediction output (ignore attention maps)
output = model(x)  # model returns (prediction, attention_maps)

dot = make_dot(output, params=dict(model.named_parameters()))
dot.render("vit_model_graph", format="png")  # Saves as 'vit_model_graph.png'
dot

## Model Graph 

In [ ]:
# num_classes=225
model = VIT_model(spec_length=intensity_points,num_output=num_classes, embed_dim=embed_dim, depth=depth, num_heads=num_heads, mlp_ratio=mlp_ratio)
summary(model)

## Validation Accuracy

In [ ]:
model.eval()
all_predicted_top1 = []
all_predicted_top3 = []
all_predicted_top5 = []
all_labels = []

with torch.no_grad():
    total = 0
    correct_top1 = 0
    correct_top3 = 0
    correct_top5 = 0

    for x_batch, y_batch in val_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(x_batch)

        # Get top-5 predictions in one call
        _, predicted_topk = outputs.topk(5, 1, largest=True, sorted=True)

        total += y_batch.size(0)

        # Compute correct predictions for Top-1, Top-3, and Top-5
        correct_top1 += (predicted_topk[:, 0] == y_batch).sum().item()
        correct_top3 += predicted_topk[:, :3].eq(y_batch.view(-1, 1)).sum().item()
        correct_top5 += predicted_topk.eq(y_batch.view(-1, 1)).sum().item()

        # Store predictions
        all_predicted_top1.extend(predicted_topk[:, 0].cpu().numpy())  # Top-1
        all_predicted_top3.extend(predicted_topk[:, :3].cpu().numpy())  # Top-3
        all_predicted_top5.extend(predicted_topk.cpu().numpy())  # Top-5
        all_labels.extend(y_batch.cpu().numpy())

    # Compute accuracy
    print(f'Top-1 Accuracy: {correct_top1 / total * 100:.2f}%')
    print(f'Top-3 Accuracy: {correct_top3 / total * 100:.2f}%')
    print(f'Top-5 Accuracy: {correct_top5 / total * 100:.2f}%')

In [ ]:
# model.eval()
# all_predicted_top1 = []
# all_predicted_top5 = []
# all_labels = []
# with torch.no_grad():
#     total = 0
#     correct_top1 = 0
#     correct_top5 = 0
#     for x_batch, y_batch in test_loader:
#         x_batch = x_batch.to(device)
#         y_batch = y_batch.to(device)

#         outputs = model(x_batch)
#         _, predicted_top1 = torch.max(outputs, 1)
#         _, predicted_top5 = outputs.topk(5, 1, True, True)

#         total += y_batch.size(0)
#         correct_top1 += (predicted_top1 == y_batch).sum().item()
#         correct_top5 += predicted_top5.eq(y_batch.view(-1, 1).expand_as(predicted_top5)).sum().item()

#         all_predicted_top1.extend(predicted_top1.cpu().numpy())
#         all_predicted_top5.extend(predicted_top5.cpu().numpy())
#         all_labels.extend(y_batch.cpu().numpy())

#     print(f'Top-1 Accuracy: {correct_top1 / total * 100}%')
#     print(f'Top-5 Accuracy: {correct_top5 / total * 100}%')




In [ ]:
# for i in range(len(all_labels)):
#     true_label = label_encoder.inverse_transform([all_labels[i]])[0]
#     predicted_label_top1 = label_encoder.inverse_transform([all_predicted_top1[i]])[0]
#     predicted_label_top5 = label_encoder.inverse_transform(all_predicted_top5[i])
#     print("True labels: ", true_label)
#     print("Predicted labels (Top-1): ", predicted_label_top1)
#     print("Predicted labels (Top-5): ", predicted_label_top5)


In [ ]:
for i in range(len(all_labels)):
    true_label = all_labels[i] + 1
    predicted_label_top1 = all_predicted_top1[i] + 1
    predicted_label_top5 = [pred + 1 for pred in all_predicted_top5[i]]

    print("True label: ", true_label)
    print("Predicted label (Top-1): ", predicted_label_top1)
    print("Predicted labels (Top-5): ", predicted_label_top5)

## Test Accuracy

In [ ]:
#data_real = pd.read_csv(''test_file_here''.csv')

#X1 = data_real.iloc[:, 1:].values
#y1 = data_real.iloc[:, 0].values

In [ ]:
#y_encoded = label_encoder.transform(y1)

#X1 = torch.tensor(X1, dtype=torch.float)
#y1 = torch.tensor(y_encoded, dtype=torch.long)

#batch_size = 64
#test_data1 = TensorDataset(X1, y1)
#test_loader_real = DataLoader(test_data1, batch_size=batch_size, shuffle=False)

In [ ]:
all_predicted_top1 = []
all_predicted_top3 = []
all_predicted_top5 = []
all_labels = []
with torch.no_grad():
    total = 0
    correct_top1 = 0
    correct_top3 = 0
    correct_top5 = 0
    for x_batch, y_batch in test_loader:

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(x_batch)
        _, predicted_top1 = torch.max(outputs, 1)
        _, predicted_top3 = outputs.topk(3, 1, True, True)
        _, predicted_top5 = outputs.topk(5, 1, True, True)

        total += y_batch.size(0)
        correct_top1 += (predicted_top1 == y_batch).sum().item()
        correct_top3 += predicted_top3.eq(y_batch.view(-1, 1).expand_as(predicted_top3)).sum().item()
        correct_top5 += predicted_top5.eq(y_batch.view(-1, 1).expand_as(predicted_top5)).sum().item()


        all_predicted_top1.extend(predicted_top1.cpu().numpy())
        all_predicted_top3.extend(predicted_top3.cpu().numpy())
        all_predicted_top5.extend(predicted_top5.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

    print(f'Top-1 Accuracy: {correct_top1 / total * 100}%')
    print(f'Top-3 Accuracy: {correct_top3 / total * 100}%')
    print(f'Top-5 Accuracy: {correct_top5 / total * 100}%')



In [ ]:

# for i in range(len(all_labels)):
#     true_label = label_encoder.inverse_transform([all_labels[i]])[0]
#     predicted_label_top1 = label_encoder.inverse_transform([all_predicted_top1[i]])[0]
#     predicted_label_top3 = label_encoder.inverse_transform(all_predicted_top3[i].tolist())
#     predicted_label_top5 = label_encoder.inverse_transform(all_predicted_top5[i].tolist())


#     print("------------")
#     print("True labels: ", true_label)
#     print("Predicted labels (Top-1): ", predicted_label_top1)
#     print("Predicted labels (Top-3): ", predicted_label_top3)
#     print("Predicted labels (Top-5): ", predicted_label_top5)
#     if true_label not in predicted_label_top1:
#         print("top 1 wrong")

#     if true_label not in predicted_label_top3:
#         print("top 3 wrong")

#     if true_label not in predicted_label_top5:
#         print("top 5 wrong")


In [ ]:
for i in range(len(all_labels)):
    true_label = all_labels[i] + 1
    predicted_label_top1 = all_predicted_top1[i] + 1
    predicted_label_top3 = [pred + 1 for pred in all_predicted_top3[i].tolist()]
    predicted_label_top5 = [pred + 1 for pred in all_predicted_top5[i].tolist()]

    print("------------")
    print("True label: ", true_label)
    print("Predicted label (Top-1): ", predicted_label_top1)
    print("Predicted labels (Top-3): ", predicted_label_top3)
    print("Predicted labels (Top-5): ", predicted_label_top5)

    if true_label not in [predicted_label_top1]:
        print("Top-1 wrong")

    if true_label not in predicted_label_top3:
        print("Top-3 wrong")

    if true_label not in predicted_label_top5:
        print("Top-5 wrong")

## Model Save

In [ ]:
save_path = f"{model_path}{datasource}_{intensity_points}_{twotheta}_{space}_{num_epochs}_{batch_size}_{embed_dim}_{num_heads}{mlp_ratio}_model_checkpoint.pth"
torch.save(model.state_dict(), save_path)